# Rehab-It RAG System

# Part 1: Build the RAG Pipeline

Our pipeline: Data -> Embeddings -> Retrieval -> LLM -> Output

In [1]:
import re
import time

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

pd.set_option("display.max_colwidth", 120)

W1109 23:24:42.273000 93477 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1.1 Knowledge Base

Our app is about exercise recovery, so our knowledge base is a guide to rehab exercises. We made it from the exercises
in our exercise list (`exercises.csv`). We grouped the exercises by body area (lower back, shoulder, knee, hip, etc.)
and put 2 exercises that go together in each paragraph, so each paragraph is about one topic.

We left out the first 8 exercises in the csv (Burpees, Crunches, etc.) because those were copied from websites and had
stuff like "YouTube·Cleveland Clinic +1" mixed into the text.

The file is `knowledge_base.txt`. Each section starts with a `#` title and paragraphs are separated by a blank line.

In [2]:
text = open("knowledge_base.txt").read()

paragraphs = []
section = ""

for block in text.split("\n\n"):
    block = block.strip()
    if block == "":
        continue
    if block.startswith("#"):
        section = block.replace("#", "").strip()
    else:
        paragraphs.append({"section": section, "text": block})

wordCounts = [len(p["text"].split()) for p in paragraphs]

print("paragraphs:", len(paragraphs))
print("shortest paragraph:", min(wordCounts), "words")
print("longest paragraph:", max(wordCounts), "words")
print("total words:", sum(wordCounts))

paragraphs: 23
shortest paragraph: 89 words
longest paragraph: 126 words
total words: 2497


In [3]:
pd.DataFrame({
    "Section": [p["section"] for p in paragraphs],
    "Words": wordCounts,
    "Paragraph": [p["text"] for p in paragraphs],
})

,Section,Words,Paragraph
0,Lower Back and Core,116,Pelvic Tilt: Lie on your back with your knees bent and feet flat on the floor. Gently flatten your lower back agains...
1,Lower Back and Core,109,"Cat-Cow Stretch: Start on your hands and knees with a flat back. Slowly round your spine toward the ceiling, then lo..."
2,Lower Back and Core,126,Seated Lower Back Rotation Stretch: Sit tall in a chair with both feet flat on the floor. Place one hand on the oppo...
3,Lower Back and Core,120,Dead Bug: Lie on your back with your arms pointing toward the ceiling and your knees bent over your hips. Slowly low...
4,Lower Back and Core,109,Glute Bridge: Lie on your back with your knees bent and feet flat. Press through your heels to lift your hips until ...
5,Shoulder and Upper Body,125,Pendulum Swing: Lean forward with one hand resting on a table and let the other arm hang straight down. Gently swing...
6,Shoulder and Upper Body,124,Wall Slide: Stand with your back and forearms against a wall. Slowly slide your arms upward while keeping your forea...
7,Shoulder and Upper Body,103,Band External Rotation: Hold a resistance band with your elbow tucked at your side and bent to 90 degrees. Slowly ro...
8,Shoulder and Upper Body,114,Wall Push-Up: Stand an arm's length from a wall and place your palms on it at shoulder height. Bend your elbows to b...
9,Shoulder and Upper Body,106,"Overhead Dumbbell Press: Hold a light dumbbell in each hand at shoulder height. Press the weights straight overhead,..."


This meets the requirements: 23 paragraphs (minimum 20), each one is between 80 and 200 words, and there are about
2,500 words total.

## 1.2 Text Representation & Chunking

**How we define a paragraph:** a block of text separated by a blank line in `knowledge_base.txt`. Every paragraph is
under a section title (body area), and we keep track of which section it came from.

To make sure we never cut a sentence in half, we split text into sentences first (at a `.`, `!` or `?` followed by a
space) and only ever put whole sentences into chunks.

In [4]:
def splitSentences(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if s.strip() != ""]


print(splitSentences(paragraphs[0]["text"])[:3])

['Pelvic Tilt: Lie on your back with your knees bent and feet flat on the floor.', 'Gently flatten your lower back against the floor by tightening your stomach muscles, then relax.', 'This slow movement eases stiffness in the lower back and wakes up the deep core muscles.']


### Strategy 1: Fixed Length Chunking

Each chunk is about 50 words. We keep adding whole sentences to a chunk until the next sentence would go over 50
words, then we start a new chunk. We do this inside each paragraph, so a chunk never mixes two different topics.

In [5]:
chunkSize = 50


def fixedChunks(paragraphs):
    chunks = []
    for p in paragraphs:
        current = []
        for sentence in splitSentences(p["text"]):
            currentWords = len(" ".join(current).split())
            if current and currentWords + len(sentence.split()) > chunkSize:
                chunks.append({"section": p["section"], "text": " ".join(current)})
                current = []
            current.append(sentence)
        if current:
            chunks.append({"section": p["section"], "text": " ".join(current)})
    return chunks

### Strategy 2: Overlapping Paragraph Chunking

Each chunk is 2 paragraphs next to each other, and every paragraph shows up in 2 chunks:

- Chunk 1 = Paragraph 1 + Paragraph 2
- Chunk 2 = Paragraph 2 + Paragraph 3
- and so on

**How the overlap works:** we move forward one paragraph at a time but each chunk is 2 paragraphs long, so neighboring
chunks share a whole paragraph (50% overlap). We only pair paragraphs from the same section, so we don't mix things
like knee exercises with neck exercises. If a section only has 1 paragraph, that paragraph is its own chunk.

In [6]:
def overlapChunks(paragraphs):
    chunks = []
    sections = []
    for p in paragraphs:
        if p["section"] not in sections:
            sections.append(p["section"])

    for section in sections:
        sectionParagraphs = [p["text"] for p in paragraphs if p["section"] == section]
        if len(sectionParagraphs) == 1:
            chunks.append({"section": section, "text": sectionParagraphs[0]})
        for i in range(len(sectionParagraphs) - 1):
            chunks.append({"section": section, "text": sectionParagraphs[i] + " " + sectionParagraphs[i + 1]})
    return chunks

### Strategy 3: Hybrid Chunking (our own strategy)

For our app, the most useful "complete thought" is one exercise: its name, how to do it, what it helps with, and what
equipment it needs. So our hybrid strategy splits each paragraph into its exercises (each exercise starts with
`Name:`) and adds the section title at the front of every chunk, like:

`Knee and Legs - Heel Slide: Lie on your back with your legs straight...`

Adding the section title gives the embedding model extra context about the body area, even if the exercise
description doesn't say it.

In [7]:
def hybridChunks(paragraphs):
    chunks = []
    for p in paragraphs:
        exercises = re.split(r"(?<=[.!?])\s+(?=[A-Z][\w\-' ]+: )", p["text"])
        for exercise in exercises:
            chunks.append({"section": p["section"], "text": p["section"] + " - " + exercise.strip()})
    return chunks

In [8]:
chunkSets = {
    "Fixed Length": fixedChunks(paragraphs),
    "Overlapping Paragraph": overlapChunks(paragraphs),
    "Hybrid": hybridChunks(paragraphs),
}

for strategy in chunkSets:
    chunks = chunkSets[strategy]
    sizes = [len(c["text"].split()) for c in chunks]
    print(strategy + ":", len(chunks), "chunks, average", round(sum(sizes) / len(sizes)), "words")

Fixed Length: 66 chunks, average 38 words
Overlapping Paragraph: 18 chunks, average 206 words
Hybrid: 46 chunks, average 58 words


Here's an example chunk from each strategy:

In [9]:
for strategy in chunkSets:
    print(strategy)
    print(chunkSets[strategy][2]["text"])
    print()

Fixed Length
This gentle stretch relieves tightness in the lower back and hips. Move slowly and stop if you feel sharp pain. No equipment is needed.

Overlapping Paragraph
Seated Lower Back Rotation Stretch: Sit tall in a chair with both feet flat on the floor. Place one hand on the opposite knee and slowly turn your upper body toward the other side. Hold for 15 seconds and switch sides. It is a gentle way to loosen the lower back during a long day of sitting, and a sturdy chair is the only thing you need. Bird Dog: Begin on your hands and knees with a flat back. Extend your right arm forward and your left leg backward at the same time, hold for a few seconds, then switch sides. The exercise strengthens the lower back, core, and hips while training balance. Keep your hips level and avoid arching your back. No equipment is needed. Dead Bug: Lie on your back with your arms pointing toward the ceiling and your knees bent over your hips. Slowly lower the opposite arm and leg toward the flo

## 1.3 Embedding Pipeline

We used 3 sentence-transformers embedding models from Hugging Face:

| Size | Model | Dimensions |
|---|---|---|
| Small | sentence-transformers/all-MiniLM-L6-v2 | 384 |
| Medium | BAAI/bge-base-en-v1.5 | 768 |
| Large | BAAI/bge-large-en-v1.5 | 1024 |

We turn every chunk into a vector with each model. That's 3 models x 3 chunking strategies = 9 sets of embeddings.
We store them in a dictionary where the key is (model, strategy) and the value is a numpy array with one row per
chunk. Our dataset is small so a numpy array is enough, we don't need a vector database.

We use `normalize_embeddings=True` so every vector has a length of 1. That way the dot product of two vectors is the
same as their cosine similarity.

In [10]:
embeddingModels = {
    "Small": "sentence-transformers/all-MiniLM-L6-v2",
    "Medium": "BAAI/bge-base-en-v1.5",
    "Large": "BAAI/bge-large-en-v1.5",
}

loadedModels = {}
for size in embeddingModels:
    loadedModels[size] = SentenceTransformer(embeddingModels[size])
    print(size, embeddingModels[size], "->", loadedModels[size].get_embedding_dimension(), "dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Small sentence-transformers/all-MiniLM-L6-v2 -> 384 dimensions


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Medium BAAI/bge-base-en-v1.5 -> 768 dimensions


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Large BAAI/bge-large-en-v1.5 -> 1024 dimensions


In [11]:
vectorStore = {}

for size in loadedModels:
    for strategy in chunkSets:
        texts = [c["text"] for c in chunkSets[strategy]]
        start = time.time()
        vectorStore[(size, strategy)] = loadedModels[size].encode(texts, normalize_embeddings=True)
        seconds = time.time() - start
        print(size, "|", strategy, "|", vectorStore[(size, strategy)].shape, "|", round(seconds, 2), "sec")

Small | Fixed Length | (66, 384) | 0.69 sec
Small | Overlapping Paragraph | (18, 384) | 0.07 sec
Small | Hybrid | (46, 384) | 0.06 sec


Medium | Fixed Length | (66, 768) | 0.22 sec


Medium | Overlapping Paragraph | (18, 768) | 0.34 sec


Medium | Hybrid | (46, 768) | 0.25 sec


Large | Fixed Length | (66, 1024) | 0.91 sec


Large | Overlapping Paragraph | (18, 1024) | 1.1 sec


Large | Hybrid | (46, 1024) | 0.78 sec


## 1.4 Retrieval System

Steps:
1. Turn the query into an embedding with the same model we used for the chunks
2. Compare it to every chunk with cosine similarity (dot product, since the vectors are normalized)
3. Return the top k chunks with the highest scores

The BGE models are trained to work better when the query starts with
"Represent this sentence for searching relevant passages: ", so we add that for those 2 models.

In [12]:
topK = 3
bgePrefix = "Represent this sentence for searching relevant passages: "


def retrieve(query, size, strategy, k=topK):
    queryText = query
    if "bge" in embeddingModels[size]:
        queryText = bgePrefix + query

    queryVector = loadedModels[size].encode([queryText], normalize_embeddings=True)[0]
    scores = vectorStore[(size, strategy)] @ queryVector
    topIndexes = np.argsort(scores)[::-1][:k]

    results = []
    for i in topIndexes:
        results.append({"score": round(float(scores[i]), 3), "text": chunkSets[strategy][i]["text"]})
    return results

Testing it with one query:

In [13]:
testQuery = "Gentle stretches for a stiff lower back"

for result in retrieve(testQuery, "Medium", "Hybrid"):
    print(result["score"], "-", result["text"][:150])

0.753 - Lower Back and Core - Cat-Cow Stretch: Start on your hands and knees with a flat back. Slowly round your spine toward the ceiling, then lower your bel
0.729 - Lower Back and Core - Pelvic Tilt: Lie on your back with your knees bent and feet flat on the floor. Gently flatten your lower back against the floor 
0.726 - Lower Back and Core - Knee-to-Chest Stretch: Lie on your back and slowly pull one knee toward your chest while keeping the other foot on the floor. Ho


## 1.5 Generation Pipeline

For the generation model we used **Qwen/Qwen2.5-0.5B-Instruct**, which we tested in A7 (it had the best balance of
speed and quality, 9.2/10). We use this same model for every experiment so the only things that change are the
embedding model and chunking strategy.

The prompt has 3 parts:
1. **System instruction:** tells the model it's the Rehab-It assistant and to only use the context
2. **Retrieved context:** the top 3 chunks from the retriever
3. **User query:** the question

We set `do_sample=False` so the model gives the same answer every time we run it, which makes the experiments fair.

In [14]:
generatorName = "Qwen/Qwen2.5-0.5B-Instruct"

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(generatorName)
generator = AutoModelForCausalLM.from_pretrained(generatorName, dtype="auto").to(device)
print("loaded", generatorName, "on", device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-0.5B-Instruct on mps


In [15]:
systemInstruction = (
    "You are the assistant for Rehab-It, an exercise recovery app. "
    "Answer the question using only the exercises in the context. "
    "Say the names of the exercises you recommend and briefly explain how to do them. "
    "If the context does not answer the question, say you don't know. "
    "Do not diagnose injuries."
)


def buildPrompt(query, retrieved):
    context = ""
    for i in range(len(retrieved)):
        context = context + "[" + str(i + 1) + "] " + retrieved[i]["text"] + "\n"

    messages = [
        {"role": "system", "content": systemInstruction},
        {"role": "user", "content": "Context:\n" + context + "\nQuestion: " + query},
    ]
    return messages


def generate(messages, maxTokens=200):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(device)
    outputs = generator.generate(**inputs, max_new_tokens=maxTokens, do_sample=False)
    newTokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(newTokens, skip_special_tokens=True).strip()

This function runs the whole pipeline (query -> retrieval -> prompt -> LLM -> answer) and times it. We'll use it in
Part 2 for all 9 configurations.

In [16]:
def askRag(query, size, strategy):
    start = time.time()
    retrieved = retrieve(query, size, strategy)
    retrievalTime = time.time() - start

    start = time.time()
    answer = generate(buildPrompt(query, retrieved))
    generationTime = time.time() - start

    return {
        "retrieved": retrieved,
        "answer": answer,
        "retrievalTime": round(retrievalTime, 3),
        "generationTime": round(generationTime, 2),
    }

Here's the full pipeline on one example so you can see the prompt and the output:

In [17]:
retrieved = retrieve(testQuery, "Medium", "Hybrid")
messages = buildPrompt(testQuery, retrieved)

print("SYSTEM INSTRUCTION:")
print(messages[0]["content"])
print()
print("USER MESSAGE (context + query):")
print(messages[1]["content"])

SYSTEM INSTRUCTION:
You are the assistant for Rehab-It, an exercise recovery app. Answer the question using only the exercises in the context. Say the names of the exercises you recommend and briefly explain how to do them. If the context does not answer the question, say you don't know. Do not diagnose injuries.

USER MESSAGE (context + query):
Context:
[1] Lower Back and Core - Cat-Cow Stretch: Start on your hands and knees with a flat back. Slowly round your spine toward the ceiling, then lower your belly and lift your chest. Repeat in a smooth rhythm for 10 breaths. This gentle spinal movement loosens a stiff lower back and can be done at home on a mat or carpet.
[2] Lower Back and Core - Pelvic Tilt: Lie on your back with your knees bent and feet flat on the floor. Gently flatten your lower back against the floor by tightening your stomach muscles, then relax. This slow movement eases stiffness in the lower back and wakes up the deep core muscles. It can be done at home on a firm 

In [18]:
generate(messages, maxTokens=10)

result = askRag(testQuery, "Medium", "Hybrid")

print("ANSWER:")
print(result["answer"])
print()
print("retrieval time:", result["retrievalTime"], "sec")
print("generation time:", result["generationTime"], "sec")

ANSWER:
For a gentle stretch for a stiff lower back, I recommend performing the **Cat-Cow Stretch** from [1]. Here's how to do it:

1. Start on your hands and knees with a flat back.
2. Slowly round your spine toward the ceiling, then lower your belly and lift your chest.
3. Repeat in a smooth rhythm for 10 breaths.

This gentle spinal movement helps loosen a stiff lower back and can be done at home on a mat or carpet.

retrieval time: 0.036 sec
generation time: 2.37 sec


The first `generate` call is just a warm-up so the first timing isn't slower than the rest.